
# NeoLM — Evaluation Summary

Consolidated evaluation results for [NeoLM-Small-Base](https://huggingface.co/MaagDeveloper/NeoLM-Small-Base), a 61.36M-parameter decoder-only Transformer pretrained from scratch.

## Evaluations

1. **Language Modeling:** Test Loss, Perplexity, and Bits per Byte.
2. **Text Generation:** Sampling configurations, EOS behavior, and generation performance.
3. **HellaSwag:** Zero-shot commonsense continuation accuracy.

Results are loaded from previously saved JSON files. No model inference or training is performed.

## Limitations

- Generation quality requires qualitative evaluation.
- HellaSwag uses a custom scoring implementation.
- Results should be interpreted relative to model size, training data, and evaluation methodology.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")

def load_results(filename):
    path = RESULTS_DIR / filename
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


ppl = load_results("neolm_perplexity.json")
generation = load_results("neolm_generation.json")
hellaswag = load_results("neolm_hellaswag.json")

generation_df = pd.DataFrame(generation["results"])

print("=" * 60)
print("NEOLM — EVALUATION RESULTS LOADED")
print("=" * 60)
print(f"Model:            {ppl['model']}")
print(f"Perplexity:       {ppl['perplexity']:.4f}")
print(f"Bits per Byte:    {ppl['bits_per_byte']:.4f}")
print(f"HellaSwag Raw:    {hellaswag['raw_accuracy']:.2%}")
print(f"HellaSwag Norm:   {hellaswag['normalized_accuracy']:.2%}")
print(f"Generations:      {len(generation_df)}")
print("=" * 60)

In [ ]:
summary = generation_df.groupby(
    "config_name",
    sort=False,
).agg(
    generations=("generated_tokens", "size"),
    eos_stops=("stopped_by_eos", "sum"),
    avg_tokens=("generated_tokens", "mean"),
    avg_tokens_per_sec=("tokens_per_second", "mean"),
).round(2)

summary["eos_rate"] = (
    summary["eos_stops"] / summary["generations"] * 100
).round(2)

print("=" * 60)
print("NEOLM — GENERATION CONFIGURATION COMPARISON")
print("=" * 60)

display(summary)

In [ ]:

metrics = {
    "Random Baseline": 25.0,
    "Raw Accuracy": hellaswag["raw_accuracy"] * 100,
    "Normalized Accuracy": hellaswag["normalized_accuracy"] * 100,
}

plt.figure(figsize=(9, 4.5))

bars = plt.bar(
    metrics.keys(),
    metrics.values(),
)

plt.bar_label(
    bars,
    fmt="%.2f%%",
    padding=4,
)

plt.axhline(
    y=25,
    linestyle="--",
    alpha=0.5,
    label="Random Chance (25%)",
)

plt.ylabel("Accuracy (%)")
plt.title("NeoLM — HellaSwag Zero-Shot Evaluation")
plt.ylim(0, 40)
plt.legend()
plt.tight_layout()
plt.show()


## Research Conclusion

### Model Overview

**NeoLM-Small-Base** is a 61.36M-parameter dense decoder-only Transformer pretrained from scratch on approximately 1.15B tokens.

### Language Modeling

| Metric | Result |
|---|---:|
| Test Loss | 3.2488 |
| Perplexity | 25.7605 |
| Bits per Byte | 0.9711 |

The evaluation successfully reproduced the previously published perplexity result.

### HellaSwag

| Metric | Accuracy |
|---|---:|
| Random Baseline | 25.00% |
| Raw Accuracy | 28.03% |
| Normalized Accuracy | 29.37% |

The model performs modestly above random chance, indicating limited commonsense continuation capabilities.

Results use a custom scoring implementation and are not directly comparable to official leaderboard scores.

### Text Generation

Qualitative evaluation revealed:

- Generally readable English and recognizable text structures.
- Some coherent short-form continuations.
- Frequent factual inaccuracies and hallucinations.
- Topic drift and occasional repetition.
- Limited code generation capabilities.
- Sensitivity to sampling configuration.

Generation quality has not been assigned a standardized human evaluation score.

### Overall Assessment

NeoLM demonstrates a successfully implemented and reproducible end-to-end language model training pipeline.

The released model learns meaningful linguistic patterns but remains limited in factual knowledge, commonsense reasoning, and long-range coherence.

These findings reflect the evaluated 61M release, not the potential performance of larger models using the same architecture.

### Next Steps

Compare NeoLM against similarly sized pretrained base language models using a consistent, lightweight evaluation protocol.
